In [ ]:
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

In [ ]:
!pip install -q youtube-transcript-api langchain-community chromadb faiss-cpu tiktoken pypdf langchain_huggingface python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/

### Input:

```
city="Bhubaneswar",
profile="Student",
activity="Outdoor walking",
duration_minutes=30
```


### Output
```
City: Bhubaneswar
PM2.5: ...
PM10: ...
Weather: ...
AQI Category: ...
Dominant Pollutant: ...
Recommendation: ...
Reason: ...
```

## Fetch Lat & Lon from city name

In [ ]:
import requests
import pandas as pd

In [ ]:
def get_coordinates(city,country_code="IN"):
  url="https://geocoding-api.open-meteo.com/v1/search"
  params= {
      "name": city,
      "count": 10,
      "language": "en",
      "format": "json",
      "countryCode": country_code
  }
  response=requests.get(url,params=params)
  data=response.json()

  if 'results' not in data:
    return None

  data=data['results'][0]

  return {
      "city": data['name'],
      "country": data['country'],
      "latitude": data['latitude'],
      "longitude": data['longitude']
  }

In [ ]:
location=get_coordinates("Bhubaneswar","IN")
location

{'city': 'Bhubaneswar',
 'country': 'India',
 'latitude': 20.27241,
 'longitude': 85.83385}

## Fetch air quality for the lat and lon

In [ ]:
def get_air_quality(lat,lon):
  url="https://air-quality-api.open-meteo.com/v1/air-quality"

  params= {
        "latitude": lat,
        "longitude": lon,
        "current": "pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,us_aqi,european_aqi",
        "hourly": "pm10,pm2_5,us_aqi,european_aqi",
        "timezone": "auto"
    }

  response=requests.get(url,params=params)
  data=response.json()
  return data["current"]

In [ ]:
pollution=get_air_quality(20.27241,85.83385)
pollution

{'time': '2026-06-14T22:30',
 'interval': 3600,
 'pm10': 80.3,
 'pm2_5': 70.3,
 'carbon_monoxide': 690.0,
 'nitrogen_dioxide': 34.7,
 'sulphur_dioxide': 13.5,
 'ozone': 61.0,
 'us_aqi': 112,
 'european_aqi': 72}

## Fetch weather data

In [ ]:
def get_weather(lat,lon):
  url="https://api.open-meteo.com/v1/forecast"

  params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation,weather_code",
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation",
        "timezone": "auto"
    }
  response=requests.get(url,params=params)
  data=response.json()
  return data["current"]

In [ ]:
weather=get_weather(20.27241,85.83385)
weather

{'time': '2026-06-14T22:30',
 'interval': 900,
 'temperature_2m': 28.1,
 'relative_humidity_2m': 91,
 'wind_speed_10m': 4.9,
 'precipitation': 0.0,
 'weather_code': 95}

## Fetch the needfull among the 3 data

In [ ]:
def extract_curr_data(location,pollution,weather):
  return {
        "city": location["city"],
        "country": location["country"],
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "time": pollution["time"],
        "pm2_5": pollution["pm2_5"],
        "pm10": pollution["pm10"],
        "us_aqi": pollution["us_aqi"],
        "european_aqi": pollution["european_aqi"],
        "carbon_monoxide": pollution["carbon_monoxide"],
        "nitrogen_dioxide": pollution["nitrogen_dioxide"],
        "sulphur_dioxide": pollution["sulphur_dioxide"],
        "ozone": pollution["ozone"],
        "temperature": weather["temperature_2m"],
        "humidity": weather["relative_humidity_2m"],
        "wind_speed": weather["wind_speed_10m"],
        "precipitation": weather["precipitation"]
    }

In [ ]:
data=extract_curr_data(location,pollution,weather)
data

{'city': 'Bhubaneswar',
 'country': 'India',
 'latitude': 20.27241,
 'longitude': 85.83385,
 'time': '2026-06-14T22:30',
 'pm2_5': 70.3,
 'pm10': 80.3,
 'us_aqi': 112,
 'european_aqi': 72,
 'carbon_monoxide': 690.0,
 'nitrogen_dioxide': 34.7,
 'sulphur_dioxide': 13.5,
 'ozone': 61.0,
 'temperature': 28.1,
 'humidity': 91,
 'wind_speed': 4.9,
 'precipitation': 0.0}

## Basic AQI Rules

In [ ]:
def get_pm25_category(pm25):
    if pm25 is None:
        return "Unknown"
    if pm25 <= 30:
        return "Good"
    elif pm25 <= 60:
        return "Satisfactory"
    elif pm25 <= 90:
        return "Moderately Polluted"
    elif pm25 <= 120:
        return "Poor"
    elif pm25 <= 250:
        return "Very Poor"
    else:
        return "Severe"


def get_pm10_category(pm10):
    if pm10 is None:
        return "Unknown"
    if pm10 <= 50:
        return "Good"
    elif pm10 <= 100:
        return "Satisfactory"
    elif pm10 <= 250:
        return "Moderately Polluted"
    elif pm10 <= 350:
        return "Poor"
    elif pm10 <= 430:
        return "Very Poor"
    else:
        return "Severe"

In [ ]:
RANK = {
    "Unknown": 0,
    "Good": 1,
    "Satisfactory": 2,
    "Moderately Polluted": 3,
    "Poor": 4,
    "Very Poor": 5,
    "Severe": 6
}

In [ ]:
def get_overall_category(pm25, pm10):
    pm25_category=get_pm25_category(pm25)
    pm10_category=get_pm10_category(pm10)

    if RANK[pm25_category]>=RANK[pm10_category]:
        return {
            "overall_category": pm25_category,
            "dominant_pollutant": "PM2.5",
            "pm25_category": pm25_category,
            "pm10_category": pm10_category
        }
    else:
        return {
            "overall_category": pm10_category,
            "dominant_pollutant": "PM10",
            "pm25_category": pm25_category,
            "pm10_category": pm10_category
        }

In [ ]:
result=get_overall_category(data["pm2_5"],data["pm10"])
result

{'overall_category': 'Moderately Polluted',
 'dominant_pollutant': 'PM2.5',
 'pm25_category': 'Moderately Polluted',
 'pm10_category': 'Satisfactory'}

In [ ]:
def generate_basic_recommendation(category,profile,activity):
    category=category.lower()
    activity=activity.lower()
    profile=profile.lower()

    if category in ["good","satisfactory"]:
        return "Outdoor activity is generally okay. Prefer low-traffic areas if possible."

    elif category=="moderately polluted":
        if "jog" in activity or "run" in activity or "cycling" in activity:
            return "You can go outside, but avoid long or intense outdoor exercise."
        else:
            return "Short outdoor activity is acceptable, but avoid staying outside for too long."

    elif category=="poor":
        if "jog" in activity or "run" in activity or "cycling" in activity:
            return "Avoid outdoor high-intensity exercise right now. Prefer indoor activity."
        else:
            return "Limit outdoor exposure and avoid polluted roads or traffic-heavy areas."

    elif category=="very poor":
        return "Avoid unnecessary outdoor activity. Keep outdoor exposure short."

    elif category=="severe":
        return "Avoid outdoor activity unless necessary."

    else:
        return "Air quality data is unavailable, so a reliable recommendation cannot be generated."

In [ ]:
suggestion=generate_basic_recommendation(result["overall_category"],"Student","Outdoor walking")
suggestion

'Short outdoor activity is acceptable, but avoid staying outside for too long.'

## Final Function

In [ ]:
def get_air_quality_plan(city,profile,activity,duration_minutes):
    location=get_coordinates(city)

    if location is None:
        return {
            "error": "City not found. Please try another city name."
        }

    pollution=get_air_quality(location["latitude"], location["longitude"])
    weather=get_weather(location["latitude"], location["longitude"])

    data=extract_curr_data(location,pollution,weather)

    result=get_overall_category(data["pm2_5"],data["pm10"])

    suggestion=generate_basic_recommendation(result["overall_category"],profile,activity)

    final_output = {
        "city": data["city"],
        "country": data["country"],
        "time": data["time"],
        "pm2_5": data["pm2_5"],
        "pm10": data["pm10"],
        "us_aqi": data["us_aqi"],
        "european_aqi": data["european_aqi"],
        "temperature": data["temperature"],
        "humidity": data["humidity"],
        "wind_speed": data["wind_speed"],
        "precipitation": data["precipitation"],
        "pm25_category": result["pm25_category"],
        "pm10_category": result["pm10_category"],
        "overall_category": result["overall_category"],
        "dominant_pollutant": result["dominant_pollutant"],
        "profile": profile,
        "activity": activity,
        "duration_minutes": duration_minutes,
        "recommendation": suggestion
    }

    return final_output

In [ ]:
test=get_air_quality_plan(
    city="Bhubaneswar",
    profile="Student",
    activity="Outdoor jogging",
    duration_minutes=30
)

test

{'city': 'Bhubaneswar',
 'country': 'India',
 'time': '2026-06-14T22:30',
 'pm2_5': 70.3,
 'pm10': 80.3,
 'us_aqi': 112,
 'european_aqi': 72,
 'temperature': 28.1,
 'humidity': 91,
 'wind_speed': 5.5,
 'precipitation': 0.0,
 'pm25_category': 'Moderately Polluted',
 'pm10_category': 'Satisfactory',
 'overall_category': 'Moderately Polluted',
 'dominant_pollutant': 'PM2.5',
 'profile': 'Student',
 'activity': 'Outdoor jogging',
 'duration_minutes': 30,
 'recommendation': 'You can go outside, but avoid long or intense outdoor exercise.'}